# 📈 Multi-Stock Price Prediction
## Using 6 Models from: *Hands-On ML with Scikit-Learn, Keras & TensorFlow* (Ch. 15)

> **Reference:** Chapter 15 — *Processing Sequences Using RNNs and CNNs*
>
> Models covered:
> | # | Model | Book Reference |
> |---|-------|----------------|
> | 1 | **LSTM** | Ch.15 — LSTM cell (original model) |
> | 2 | **SimpleRNN** | Ch.15 — Simple RNN / Deep RNN |
> | 3 | **GRU** | Ch.15 — GRU cell |
> | 4 | **Bidirectional LSTM** | Ch.16 — Bidirectional RNNs |
> | 5 | **CNN + GRU (WaveNet-style)** | Ch.15 — WaveNet / Conv1D + GRU |
> | 6 | **Linear Regression (Baseline)** | Ch.15 — Baseline Metrics |
>
> **Stocks:** AAPL · NVDA · MSFT · GOOGL | **Period:** 2015–2023 | **Prediction:** 1 year


## 1️⃣ Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, LSTM, GRU, SimpleRNN,
    Bidirectional, Conv1D, MaxPooling1D
)


## 2️⃣ Settings

In [ ]:
stocks      = ["AAPL", "NVDA", "MSFT", "GOOGL"]
start_date  = "2015-01-01"
end_date    = "2023-12-31"
time_step   = 60    # look-back window
future_days = 365


## 3️⃣ Helper Functions

In [ ]:
def prepare_data(df, time_step=60):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df.values)
    X, y = [], []
    for i in range(time_step, len(scaled)):
        X.append(scaled[i - time_step:i, 0])
        y.append(scaled[i, 0])
    return np.array(X), np.array(y), scaler, scaled


def train_test_split_ts(X, y, ratio=0.8):
    split = int(len(X) * ratio)
    return X[:split], X[split:], y[:split], y[split:]


def calc_metrics(real, pred):
    mae  = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    r2   = r2_score(real, pred)
    acc  = 100 - (mae / np.mean(real) * 100)
    return mae, rmse, r2, acc


---
## 🔵 Model 1: LSTM *(Original Model)*
> **📖 Book Reference — Chapter 15, p. 544:**
> *"In Keras, you can simply use the LSTM layer instead of the SimpleRNN layer"*
>
> ```python
> model = keras.models.Sequential([
>     keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]),
>     keras.layers.LSTM(20, return_sequences=True),
>     ...
> ])
> ```
> LSTM can learn to **remember long-term patterns** and forget irrelevant ones —  
> ideal for stock price sequences with trends spanning months.


In [ ]:
def build_lstm(time_step):
    model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(time_step, 1)),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


lstm_results = []

for symbol in stocks:
    print(f"\n[LSTM] {symbol} ...")
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_lstm(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred = scaler.inverse_transform(model.predict(X_test))
    real = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    lstm_results.append([symbol, "LSTM", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ LSTM done")


---
## 🟤 Model 2: Deep SimpleRNN
> **📖 Book Reference — Chapter 15, p. 536:**
> *"Implementing a deep RNN with tf.keras is quite simple: just stack recurrent layers."*
>
> ```python
> model = keras.models.Sequential([
>     keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
>     keras.layers.SimpleRNN(20, return_sequences=True),
>     keras.layers.SimpleRNN(1)
> ])
> ```
> The book uses SimpleRNN as the **starting point** before introducing LSTM/GRU.  
> Useful as a comparison — shows why LSTM beats basic RNN on long sequences.


In [ ]:
def build_simplernn(time_step):
    model = Sequential([
        SimpleRNN(50, return_sequences=True, input_shape=(time_step, 1)),
        SimpleRNN(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


simplernn_results = []

for symbol in stocks:
    print(f"\n[SimpleRNN] {symbol} ...")
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_simplernn(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred = scaler.inverse_transform(model.predict(X_test))
    real = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    simplernn_results.append([symbol, "SimpleRNN", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ SimpleRNN done")


---
## 🟢 Model 3: GRU *(Gated Recurrent Unit)*
> **📖 Book Reference — Chapter 15, p. 548–549:**
> *"The GRU cell is a simplified version of the LSTM cell, and it seems to perform just as well."*
>
> ```python
> keras.layers.GRU(layer)  # just replace SimpleRNN or LSTM with GRU
> ```
> The book explains GRU merges the forget & input gates into one — **faster training,  
> same accuracy** as LSTM in most cases.


In [ ]:
def build_gru(time_step):
    model = Sequential([
        GRU(50, return_sequences=True, input_shape=(time_step, 1)),
        GRU(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


gru_results = []

for symbol in stocks:
    print(f"\n[GRU] {symbol} ...")
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_gru(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred = scaler.inverse_transform(model.predict(X_test))
    real = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    gru_results.append([symbol, "GRU", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ GRU done")


---
## 🟣 Model 4: Bidirectional LSTM
> **📖 Book Reference — Chapter 16, p. 573 (Bidirectional RNNs):**
> *"A bidirectional RNN consists of two regular RNNs: one that reads the input forward,  
> and the other that reads it backward. This allows the network to capture both  
> past and future context."*
>
> ```python
> keras.layers.Bidirectional(keras.layers.GRU(10, return_sequences=True))
> ```
> Reads the price sequence both **forward and backward** — captures more patterns  
> and generally gives better accuracy than one-direction LSTM.


In [ ]:
def build_bilstm(time_step):
    model = Sequential([
        Bidirectional(LSTM(50, return_sequences=True), input_shape=(time_step, 1)),
        Bidirectional(LSTM(50)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


bilstm_results = []

for symbol in stocks:
    print(f"\n[BiLSTM] {symbol} ...")
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_bilstm(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred = scaler.inverse_transform(model.predict(X_test))
    real = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    bilstm_results.append([symbol, "BiLSTM", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ Bidirectional LSTM done")


---
## 🟡 Model 5: CNN + GRU *(WaveNet-style)*
> **📖 Book Reference — Chapter 15, p. 550–552 (WaveNet):**
> *"We will finish this chapter by implementing a WaveNet: a CNN architecture  
> capable of handling sequences of tens of thousands of time steps."*
>
> ```python
> model = keras.models.Sequential([
>     keras.layers.Conv1D(filters=20, kernel_size=4, strides=2, padding="valid",
>                         input_shape=[None, 1]),
>     keras.layers.GRU(20, return_sequences=True),
>     keras.layers.GRU(20, return_sequences=True),
>     ...
> ])
> ```
> **CNN** extracts local price patterns, then **GRU** learns time dependencies —  
> the exact combo the book recommends for long time series.


In [ ]:
def build_cnn_gru(time_step):
    model = Sequential([
        Conv1D(filters=64, kernel_size=4, strides=1,
               padding="causal", activation="relu",
               input_shape=(time_step, 1)),
        GRU(50, return_sequences=True),
        GRU(50),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


cnngru_results = []

for symbol in stocks:
    print(f"\n[CNN+GRU] {symbol} ...")
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = build_cnn_gru(time_step)
    model.fit(X_train, y_train, epochs=5, batch_size=32, verbose=0)

    pred = scaler.inverse_transform(model.predict(X_test))
    real = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    cnngru_results.append([symbol, "CNN+GRU", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ CNN+GRU done")


---
## 🟠 Model 6: Linear Regression *(Baseline)*
> **📖 Book Reference — Chapter 15, p. 534–535 (Baseline Metrics):**
> *"Before we start using RNNs, it is often a good idea to have a few baseline metrics,  
> or else we may end up thinking our model works great when in fact it is doing worse  
> than basic models."*
>
> ```python
> # Another simple approach: use a fully connected network
> model = keras.models.Sequential([keras.layers.Flatten(input_shape=[50, 1]),
>                                   keras.layers.Dense(1)])
> ```
> The book explicitly recommends a **simple baseline** before judging RNN performance.  
> We use Linear Regression as this baseline.


In [ ]:
lr_results = []

for symbol in stocks:
    print(f"\n[LinearReg] {symbol} ...")
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    # sklearn models use 2D X (no reshape to 3D needed)
    X_train, X_test, y_train, y_test = train_test_split_ts(X, y)

    model = LinearRegression()
    model.fit(X_train, y_train)

    pred = scaler.inverse_transform(model.predict(X_test).reshape(-1, 1))
    real = scaler.inverse_transform(y_test.reshape(-1, 1))
    mae, rmse, r2, acc = calc_metrics(real, pred)

    lr_results.append([symbol, "LinearReg (Baseline)", mae, rmse, r2, acc])
    print(f"  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  Acc={acc:.2f}%")

print("\n✅ Linear Regression done")


---
## 📊 Model Comparison Table
> All 6 models side-by-side for every stock.


In [ ]:
all_results = (
    lstm_results +
    simplernn_results +
    gru_results +
    bilstm_results +
    cnngru_results +
    lr_results
)

comparison_df = pd.DataFrame(
    all_results,
    columns=["Stock", "Model", "MAE", "RMSE", "R2", "Accuracy %"]
)

comparison_df = comparison_df.sort_values(["Stock", "Accuracy %"], ascending=[True, False])
comparison_df = comparison_df.reset_index(drop=True)

for col in ["MAE", "RMSE", "R2", "Accuracy %"]:
    comparison_df[col] = comparison_df[col].round(4)

comparison_df


## 🏆 Best Model Per Stock

In [ ]:
best_df = comparison_df.loc[
    comparison_df.groupby("Stock")["Accuracy %"].idxmax()
].reset_index(drop=True)

print("🏆 Best model for each stock:\n")
print(best_df[["Stock", "Model", "Accuracy %"]].to_string(index=False))


## 📈 Accuracy Chart — All Models vs All Stocks

In [ ]:
models     = comparison_df["Model"].unique()
stock_list = comparison_df["Stock"].unique()
x          = np.arange(len(stock_list))
width      = 0.13
colors     = ["#4C72B0","#8B5E3C","#55A868","#8B4FCF","#C4A82D","#E07B3A"]

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model, color) in enumerate(zip(models, colors)):
    accs = []
    for stock in stock_list:
        row = comparison_df[
            (comparison_df["Model"] == model) &
            (comparison_df["Stock"] == stock)
        ]["Accuracy %"]
        accs.append(row.values[0] if len(row) else 0)

    offset = (i - len(models) / 2) * width + width / 2
    ax.bar(x + offset, accs, width, label=model, color=color, alpha=0.85)

ax.set_xlabel("Stock", fontsize=12)
ax.set_ylabel("Accuracy %", fontsize=12)
ax.set_title("Model Accuracy Comparison per Stock", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(stock_list, fontsize=11)
ax.legend(title="Model", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_ylim(0, 110)
ax.axhline(y=100, color="gray", linestyle="--", alpha=0.4)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("model_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 🔮 Future Price Prediction — LSTM (1 Year)
> Same as original project: real price (blue) + future prediction (orange).


In [ ]:
for symbol in stocks:
    df = yf.download(symbol, start=start_date, end=end_date)[['Close']]
    df.dropna(inplace=True)

    X, y, scaler, scaled = prepare_data(df, time_step)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    model = build_lstm(time_step)
    model.fit(X, y, epochs=5, batch_size=32, verbose=0)

    temp   = scaled[-time_step:].tolist()
    future = []
    for _ in range(future_days):
        x_in = np.array(temp[-time_step:]).reshape(1, time_step, 1)
        p    = model.predict(x_in, verbose=0)[0][0]
        temp.append([p])
        future.append([p])

    future_prices             = scaler.inverse_transform(np.array(future))
    future_plot               = np.full((len(df) + future_days, 1), np.nan)
    future_plot[len(df):]     = future_prices

    plt.figure(figsize=(12, 4))
    plt.plot(df.values,   label="Real Price",   color="#4C72B0")
    plt.plot(future_plot, label="Future (LSTM)", color="#E07B3A")
    plt.title(f"{symbol} — Real vs Future Price (LSTM)", fontsize=13, fontweight="bold")
    plt.xlabel("Days")
    plt.ylabel("Price (USD)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


---
## 📝 Model Evaluation Summary

| Model | Book Chapter | Key Idea |
|-------|-------------|----------|
| **LSTM** | Ch.15 p.544 | Original model — learns long & short-term memory via gates |
| **SimpleRNN (Deep)** | Ch.15 p.536 | Stack RNN layers — baseline deep recurrent model |
| **GRU** | Ch.15 p.548 | Simplified LSTM — faster, same accuracy |
| **Bidirectional LSTM** | Ch.16 p.573 | Reads sequence forward & backward |
| **CNN + GRU (WaveNet)** | Ch.15 p.550 | Conv1D extracts patterns, GRU learns time dependencies |
| **Linear Regression** | Ch.15 p.534 | Baseline metric — compare before judging RNNs |

**Metrics:**
- **MAE** — Mean Absolute Error *(lower = better)*
- **RMSE** — Root Mean Square Error *(lower = better)*
- **R²** — Fit quality *(closer to 1 = better)*
- **Accuracy %** — `100 - (MAE / mean_price × 100)`

**Stocks:** AAPL · NVDA · MSFT · GOOGL | **Dataset:** Yahoo Finance | **Period:** 2015–2023
